<a href="https://colab.research.google.com/github/luciaPi/MLSS2026-generative-models/blob/main/4_Diffusion_MNIST_cisty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Difúzny Model na MNIST

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používam zariadenie: {device}")

## 1. Hyperparametre a difúzny schedule

In [ ]:
# Hyperparametre
BATCH_SIZE = 128
EPOCHS = 20
LEARNING_RATE = 1e-3
TIMESTEPS = 1000
BETA_START = 0.0001
BETA_END = 0.02

print(f"Timesteps: {TIMESTEPS}")  # počet krokov difúzneho procesu T=1000
print(f"Beta range: {BETA_START} → {BETA_END}")  # miera šumu pridaného v každom kroku (timestep) -> postupne sa zvysuje

In [ ]:
# Dáta
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Dataset size: {len(train_dataset)}")

## 2. Difúzny Schedule

In [ ]:
# plan pridavania sumu do obrazka (rychlejsie vypocitat vopred)
def create_diffusion_schedule(timesteps, beta_start, beta_end):
    betas = torch.linspace(beta_start, beta_end, timesteps) # miera šumu pridaného v každom kroku (timestep)
    # Začíname s malým šumom (beta_start) a končíme s veľkým šumom (beta_end)
    alphas = 1.0 - betas # alpha - koľko z pôvodného signálu si zachováme
    alphas_cumprod = torch.cumprod(alphas, dim=0) # kumulovane alphy - umožňuje preskočiť priamo na ľubovoľný timestep bez prechádzania všetkými krokmi
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod) # stredna hodnota pridaneho sumu (blizi sa k 0)    N(0,1)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod) # rozptyl pridaneho sumu (blizi sa k 1)    N(0,1)
    sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
    posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

    return {
        'betas': betas,
        'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'sqrt_alphas_cumprod': sqrt_alphas_cumprod,
        'sqrt_one_minus_alphas_cumprod': sqrt_one_minus_alphas_cumprod,
        'sqrt_recip_alphas': sqrt_recip_alphas,
        'posterior_variance': posterior_variance
    }

schedule = create_diffusion_schedule(TIMESTEPS, BETA_START, BETA_END)
for key in schedule:
    schedule[key] = schedule[key].to(device)

print("Difúzny schedule vytvorený!")

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(schedule['betas'].cpu())
plt.title('Beta schedule')
plt.xlabel('Timestep')
plt.ylabel('Beta')
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(schedule['alphas_cumprod'].cpu())
plt.title('Alpha cumprod')
plt.xlabel('Timestep')
plt.ylabel('Alpha cumprod')
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(schedule['sqrt_one_minus_alphas_cumprod'].cpu())
plt.title('Sqrt(1 - alpha_cumprod)')
plt.xlabel('Timestep')
plt.ylabel('Úroveň šumu')
plt.grid(True)

plt.tight_layout()
plt.show()

## 3. Forward Diffusion

In [ ]:
def q_sample(x_0, t, schedule, noise=None): # generovanie zasumeneho obrazka v case t
    if noise is None:
        noise = torch.randn_like(x_0)

    sqrt_alphas_cumprod_t = schedule['sqrt_alphas_cumprod'][t].view(-1, 1, 1, 1) # stredna hodnota
    sqrt_one_minus_alphas_cumprod_t = schedule['sqrt_one_minus_alphas_cumprod'][t].view(-1, 1, 1, 1) # rozptyl

    return sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise # generuje z normalneho rozdelenia

In [ ]:
print("="*60)
print("DEMO: Forward Diffusion Process")
print("="*60)

test_img, _ = next(iter(train_loader))
test_img = test_img[0:1].to(device)

timesteps_to_show = [0, 50, 100, 250, 500, 750, 999]

plt.figure(figsize=(14, 2))
for idx, t in enumerate(timesteps_to_show):
    t_tensor = torch.tensor([t], device=device)
    noisy = q_sample(test_img, t_tensor, schedule) # generovanie zasumeneho obrazka v case t

    plt.subplot(1, len(timesteps_to_show), idx+1)
    plt.imshow(noisy[0].cpu().squeeze(), cmap='gray')
    plt.title(f't={t}')
    plt.axis('off')

plt.suptitle('Forward Diffusion: Postupné pridávanie šumu', fontsize=14)
plt.tight_layout()
plt.show()

## 4. U-Net Model
<img src="https://towardsdatascience.com/wp-content/uploads/2024/07/1xJuCrrm_l36Kr8Xx-iGT0Q-1.png">

In [ ]:
# hovori modelu, v akom kroku (timestep) sa nachadza
# Sinusoidal Embedding
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim)
        )

    # kazdy timestep ma jedinecne zakodovanie
    def forward(self, t):
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return self.mlp(emb)

class SimpleUNet(nn.Module):
    def __init__(self, channels=1, time_dim=256):
        super().__init__()

        self.time_mlp = TimeEmbedding(time_dim)

        self.conv1 = nn.Conv2d(channels, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)

        self.time_proj1 = nn.Linear(time_dim, 64)
        self.time_proj2 = nn.Linear(time_dim, 128)
        self.time_proj3 = nn.Linear(time_dim, 256)

        self.bottleneck = nn.Conv2d(256, 256, 3, padding=1)

        self.upconv3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv4 = nn.Conv2d(256, 128, 3, padding=1)

        self.upconv2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv5 = nn.Conv2d(128, 64, 3, padding=1)

        self.upconv1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv6 = nn.Conv2d(96, 32, 3, padding=1)

        self.out = nn.Conv2d(32, channels, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t) # time embedding

        # zmensovanie rozmeru, zvacsovanie poctu priznakov
        # vstupuju aj time embeddingy
        x1 = F.relu(self.conv1(x) + self.time_proj1(t_emb)[:, :, None, None])
        x1_pool = F.max_pool2d(x1, 2)

        x2 = F.relu(self.conv2(x1_pool) + self.time_proj2(t_emb)[:, :, None, None])
        x2_pool = F.max_pool2d(x2, 2)

        x3 = F.relu(self.conv3(x2_pool) + self.time_proj3(t_emb)[:, :, None, None])

        x = F.relu(self.bottleneck(x3)) # bottleneck

        # zvacsovanie rozmeru, zmensovanie poctu priznakov
        x = self.upconv3(x)
        x = torch.cat([x, x2], dim=1)
        x = F.relu(self.conv4(x))

        x = self.upconv2(x)
        x = torch.cat([x, x1], dim=1)
        x = F.relu(self.conv5(x))

        x = self.upconv1(x)
        x1_up = F.interpolate(self.conv1(torch.randn_like(x[:,:1])), size=x.shape[2:])
        x = torch.cat([x, x1_up], dim=1)
        x = F.relu(self.conv6(x))

        x = F.interpolate(x, size=(28, 28), mode='bilinear', align_corners=False)

        return self.out(x)

model = SimpleUNet().to(device)
print(f"Model parametrov: {sum(p.numel() for p in model.parameters())}")

## 5. Trénovanie

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("="*60)
print("TRÉNOVANIE DIFÚZNEHO MODELU")
print("="*60)
print("")

losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for batch in tqdm(train_loader, desc=f"Epocha {epoch+1}/{EPOCHS}"):
        x_0, _ = batch
        x_0 = x_0.to(device)
        batch_size = x_0.size(0)

        t = torch.randint(0, TIMESTEPS, (batch_size,), device=device).long() # nahodny krok (timestep)
        noise = torch.randn_like(x_0)
        x_t = q_sample(x_0, t, schedule, noise=noise) # zasumeny obrazok v kroku t

        predicted_noise = model(x_t, t) # model predikuje sum v kroku t
        loss = F.mse_loss(predicted_noise, noise) # porovnane so skutocnym sumom

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)

    print(f"Epocha {epoch+1}/{EPOCHS} | Loss: {avg_loss:.6f}")

print("\nTrénovanie dokončené!")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epocha')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

## 6. Reverse Diffusion (Sampling)

In [ ]:
@torch.no_grad()
def p_sample(model, x_t, t, schedule):
    betas_t = schedule['betas'][t].view(-1, 1, 1, 1)
    sqrt_one_minus_alphas_cumprod_t = schedule['sqrt_one_minus_alphas_cumprod'][t].view(-1, 1, 1, 1)
    sqrt_recip_alphas_t = schedule['sqrt_recip_alphas'][t].view(-1, 1, 1, 1)

    predicted_noise = model(x_t, t)

    model_mean = sqrt_recip_alphas_t * (
        x_t - betas_t * predicted_noise / sqrt_one_minus_alphas_cumprod_t # odcitavame predikovany sum (skalovany)
    )

    if t[0] == 0:
        return model_mean
    else:
        posterior_variance_t = schedule['posterior_variance'][t].view(-1, 1, 1, 1)
        noise = torch.randn_like(x_t)
        return model_mean + torch.sqrt(posterior_variance_t) * noise

@torch.no_grad()
def p_sample_loop(model, shape, schedule):
    device = next(model.parameters()).device
    x = torch.randn(shape, device=device)

    for i in tqdm(reversed(range(TIMESTEPS)), desc='Sampling', total=TIMESTEPS):
        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        x = p_sample(model, x, t, schedule)

    return x

print("Sampling funkcie definované!")

## 7. Generovanie obrázkov

In [ ]:
print("="*60)
print("GENEROVANIE NOVÝCH OBRÁZKOV")
print("="*60)
print("Toto trvá dlho (1000 krokov)")
print("")

model.eval()
samples = p_sample_loop(model, (16, 1, 28, 28), schedule)

plt.figure(figsize=(8, 8))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(samples[i].cpu().squeeze(), cmap='gray')
    plt.axis('off')
plt.suptitle('Generované číslice', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Vizualizácia reverse process

In [ ]:
print("="*60)
print("VIZUALIZÁCIA REVERSE PROCESS")
print("="*60)

model.eval()
x = torch.randn(1, 1, 28, 28, device=device)

timesteps_to_show = [999, 750, 500, 250, 100, 50, 0]
images = [x.clone()]

with torch.no_grad():
    for i in tqdm(reversed(range(TIMESTEPS)), desc='Denoising', total=TIMESTEPS):
        t = torch.full((1,), i, device=device, dtype=torch.long)
        x = p_sample(model, x, t, schedule)

        if i in timesteps_to_show:
            images.append(x.clone())

plt.figure(figsize=(14, 2))
for idx, (img, t) in enumerate(zip(images, [999] + timesteps_to_show)):
    plt.subplot(1, len(images), idx+1)
    plt.imshow(img[0].cpu().squeeze(), cmap='gray')
    plt.title(f't={t}')
    plt.axis('off')

plt.suptitle('Reverse Diffusion: Postupné odstraňovanie šumu', fontsize=14)
plt.tight_layout()
plt.show()

## Záver

**Kľúčové poznatky:**

1. **Forward diffusion**: Postupne pridávame šum
2. **Reverse diffusion**: Postupne odstraňujeme šum
3. **Model predikuje ŠUM**, nie obrázok!
4. **1000 krokov** = vysoká kvalita, ale pomalé
5. **Stabilný tréning** - na rozdiel od GAN

Vytvorené s použitím Claude AI.